# Distributional similarity metrics — real vs synthetic

Numeric companion to `run_visual_checks.ipynb`: same model/config/generation pipeline,
but instead of plots it produces scalar tables using the metric functions from
`genpm.data_similarity.data_similarity_utils` (Wasserstein, MMD, Jensen-Shannon,
hourly-profile RMSE, ACF distance, Lomb-Scargle spectrum distance, plus the
cross-KPI metrics: sliced Wasserstein, multivariate MMD, pairwise/partial
correlation distance).

Point `RUN_ID` / `WEIGHTS_FILE` at any trained run and re-run top to bottom.

In [ ]:
import getpass
import random
import sys

_USER = getpass.getuser()
sys.path.insert(0, f"/home/{_USER}/app/apps/apps/generator")

import tsgm  # noqa: E402, F401  -- must be imported before keras
import keras  # noqa: E402, F401
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
from IPython.display import display  # noqa: E402

from genpm.data_similarity.data_similarity_utils import (  # noqa: E402
    acf_distance,
    acf_nan_aware,
    corr_matrix_distance,
    hourly_profile,
    hourly_profile_rmse,
    kde_curves,
    lomb_scargle_spectrum,
    metric_jensen_shannon,
    metric_mmd,
    metric_mmd_multivariate,
    metric_sliced_wasserstein,
    metric_wasserstein_1d,
    pairwise_corr_matrix_from_pdf,
    partial_correlations,
    partial_corr_distance,
    spectrum_distance,
)
from genpm.modelling.core.artifacts import load_trained_model  # noqa: E402
from genpm.modelling.core.generation import generate_windows  # noqa: E402
from genpm.modelling.validation import long_to_wide, wide_timespan_to_long  # noqa: E402
from genpm.preprocessing.logic.scaling import SimpleMinMaxScaler, YeoJohnsonScaler  # noqa: E402
from genpm.utils.consts import SHARED_DIR_PATH  # noqa: E402
from genpm.utils.spark_session import SparkDataManager  # noqa: E402


In [ ]:
# ── Point this at whichever run you want to inspect ──
RUN_ID = "diffusion_run_5_yj"
WEIGHTS_FILE = "ddpm_5_yj.weights.h5"  # or the "*_last.weights.h5" fallback

RUN_DIR = SHARED_DIR_PATH / "model_runs" / RUN_ID
WEIGHTS_PATH = RUN_DIR / "models_weights_debug" / WEIGHTS_FILE

PREPROC_DIR = SHARED_DIR_PATH / "preprocessed_dataset" / "final_pmcm_yj"

model, config_encoder, cell_config_map = load_trained_model(
    run_id_path=RUN_DIR,
    weights_path=WEIGHTS_PATH,
)

X_scaled = np.load(RUN_DIR / "X_scaled.npy")
y_all = np.load(RUN_DIR / "y.npy")
cell_ids = np.load(RUN_DIR / "cell_ids.npy", allow_pickle=True)
window_anchors = np.load(RUN_DIR / "window_anchors.npy", allow_pickle=True)
kpi_columns = np.load(RUN_DIR / "kpi_columns.npy", allow_pickle=True).tolist()

SEQ_LEN, FEAT_DIM = X_scaled.shape[1], X_scaled.shape[2]
print(f"Loaded {RUN_ID}: {len(X_scaled):,} windows, seq_len={SEQ_LEN}, feat_dim={FEAT_DIM}")


## Pick a couple of configs to compare

One example real `distname` per *distinct* cell-config combination actually present in training, same selection as `run_visual_checks.ipynb`.

In [ ]:
from collections import defaultdict

_by_combo = defaultdict(list)
for _cid, _cfg in cell_config_map["map"].items():
    _by_combo[tuple(_cfg)].append(_cid)

# label -> example real cell_id for that config
CONFIGS = {f"config_{i+1} ({combo[1]}/{combo[2]}/{combo[3]})": cids[0] for i, (combo, cids) in enumerate(_by_combo.items())}
print(f"{len(CONFIGS)} distinct configs found:")
for label, cid in CONFIGS.items():
    print(f"  {label}  ->  cell_id={cid}")


In [ ]:
EXAMPLE_KPIS = random.sample(kpi_columns, 30)
EXAMPLE_KPIS = [k for k in EXAMPLE_KPIS if k in kpi_columns]


## Helper functions — window selection / reconstruction

In [ ]:
def _to_numpy(t):
    try:
        return np.asarray(t)
    except TypeError:
        return t.detach().cpu().numpy()


def select_real_windows(cell_id: str, max_windows: int = 8, seed: int = 0):
    """Real (X, y, window_anchor) rows for one cell, capped to max_windows."""
    idx = np.where(cell_ids == cell_id)[0]
    if len(idx) == 0:
        raise ValueError(f"No windows found for cell_id={cell_id!r}")
    if len(idx) > max_windows:
        idx = np.random.default_rng(seed).choice(idx, size=max_windows, replace=False)
    return X_scaled[idx], y_all[idx], pd.to_datetime(window_anchors[idx])


def wide_from_array(arr: np.ndarray, anchors: pd.DatetimeIndex, cell_id: str) -> pd.DataFrame:
    """(n, seq_len, n_kpis) array -> wide df with cell_id/window_anchor/timestamp + KPI cols,
    same shape generate_windows() produces, so wide_timespan_to_long handles both identically."""
    n, seq_len, n_kpis = arr.shape
    flat = arr.reshape(n * seq_len, n_kpis)
    df = pd.DataFrame(flat, columns=kpi_columns[:n_kpis])
    df.insert(
        0,
        "timestamp",
        pd.to_datetime(np.repeat(anchors.values, seq_len))
        + pd.to_timedelta(np.tile(np.arange(seq_len), n), unit="h"),
    )
    df.insert(0, "window_anchor", np.repeat(anchors.values, seq_len))
    df.insert(0, "cell_id", cell_id)
    return df


## Inverse transform back to physical KPI units

Same pipeline as `run_visual_checks.ipynb` / `inverse_transform_testing.ipynb`: MinMax inverse → Yeo–Johnson inverse.

In [ ]:
sdm = SparkDataManager()
mm_params = sdm.read_parquet(PREPROC_DIR / "scaling_params_df")
yj_params = sdm.read_parquet(PREPROC_DIR / "yj_scaling_params_df")

_MM_GROUP_COLS = [c for c in mm_params.columns if c not in ("mm_min", "mm_max")]
_YJ_GROUP_COLS = [c for c in yj_params.columns if c != "yj_lambda"]

_mm_scaler = SimpleMinMaxScaler.load_params(
    mm_params, value_col="kpi_value", group_cols=_MM_GROUP_COLS
)
_yj_scaler = YeoJohnsonScaler.load_params(
    yj_params, value_col="kpi_value", group_cols=_YJ_GROUP_COLS
)
print(f"Inverse-transform params loaded — group cols: {_MM_GROUP_COLS}")


def _attach_config_cols(long_df: pd.DataFrame, cell_id: str) -> pd.DataFrame:
    """Add the five [CELL] config columns the scaler params join on, for this cell."""
    cfg_cols = cell_config_map["config_cols"]
    cfg_vals = cell_config_map["map"][str(cell_id)]
    out = long_df[["distname", "timestamp", "window_anchor", "kpi_id", "kpi_value"]].copy()
    for col, val in zip(cfg_cols, cfg_vals, strict=True):
        out[col] = val
    return out


def inverse_to_physical(long_real: pd.DataFrame, long_syn: pd.DataFrame, cell_id: str):
    """Scaled-space long frames -> original physical KPI units."""
    real = _attach_config_cols(long_real, cell_id).assign(__src__="real")
    syn = _attach_config_cols(long_syn, cell_id).assign(__src__="syn")
    combined = pd.concat([real, syn], ignore_index=True)

    sdf = sdm.spark.createDataFrame(combined)
    sdf = _mm_scaler.inverse_transform(sdf)  # undo MinMax [0,1] scaling
    sdf = _yj_scaler.inverse_transform(sdf)  # undo Yeo-Johnson
    inv = sdf.toPandas()

    keep = ["distname", "timestamp", "window_anchor", "kpi_id", "kpi_value"]
    real_phys = inv.loc[inv["__src__"] == "real", keep].reset_index(drop=True)
    syn_phys = inv.loc[inv["__src__"] == "syn", keep].reset_index(drop=True)
    return real_phys, syn_phys


## Metric helper functions

`compute_single_kpi_metrics` wraps the per-KPI metrics from `data_similarity_utils`
(Wasserstein, MMD, Jensen-Shannon, hourly-profile RMSE, ACF distance, Lomb-Scargle
spectrum distance) for a long-format real/synthetic pair. `compute_multi_kpi_metrics`
wraps the cross-KPI ones (sliced Wasserstein, multivariate MMD, pairwise/partial
correlation distance).

Real windows here are a handful of non-contiguous sampled weeks (vs. synthetic's
contiguous run), so ACF distance — which compares raw array positions, not
timestamps — is a rough signal at window boundaries; Lomb-Scargle is unaffected
since it's built for irregular sampling.

In [ ]:
def compute_single_kpi_metrics(
    long_real: pd.DataFrame,
    long_syn: pd.DataFrame,
    kpi_list: list[str],
    max_lag: int = 48,
) -> pd.DataFrame:
    """Per-KPI distributional/temporal similarity metrics, real vs synthetic."""
    rows = []
    for kpi in kpi_list:
        r_df = long_real.loc[long_real["kpi_id"] == kpi].sort_values("timestamp")
        s_df = long_syn.loc[long_syn["kpi_id"] == kpi].sort_values("timestamp")
        r_vals = r_df["kpi_value"].dropna().to_numpy()
        s_vals = s_df["kpi_value"].dropna().to_numpy()

        if r_vals.size == 0 or s_vals.size == 0:
            rows.append({"KPI": kpi})
            continue

        _, kde_r, kde_s = kde_curves(r_vals, s_vals)

        real_hourly = hourly_profile(r_df, value_col="kpi_value", ts_col="timestamp")
        synth_hourly = hourly_profile(s_df, value_col="kpi_value", ts_col="timestamp")

        # acf/spectrum need values aligned 1:1 with their own timestamps — use the
        # full (NaN-preserving) series here, not the dropna'd r_vals/s_vals above,
        # since dropping rows would desync the value array from r_df["timestamp"].
        r_full = r_df["kpi_value"].to_numpy(dtype=float)
        s_full = s_df["kpi_value"].to_numpy(dtype=float)

        acf_r = acf_nan_aware(r_full, max_lag=max_lag)
        acf_s = acf_nan_aware(s_full, max_lag=max_lag)

        per_r, pow_r = lomb_scargle_spectrum(
            ts_hours=(r_df["timestamp"] - r_df["timestamp"].min()).dt.total_seconds().to_numpy() / 3600.0,
            values=r_full,
        )
        per_s, pow_s = lomb_scargle_spectrum(
            ts_hours=(s_df["timestamp"] - s_df["timestamp"].min()).dt.total_seconds().to_numpy() / 3600.0,
            values=s_full,
        )

        rows.append(
            {
                "KPI": kpi,
                "wasserstein": metric_wasserstein_1d(r_vals, s_vals),
                "mmd": metric_mmd(r_vals, s_vals),
                "jensen_shannon": metric_jensen_shannon(kde_r, kde_s),
                "hourly_rmse": hourly_profile_rmse(real_hourly, synth_hourly),
                "acf_dist": acf_distance(acf_r, acf_s),
                "ls_spectrum_dist": spectrum_distance(pow_r, pow_s),
            }
        )

    return pd.DataFrame(rows).set_index("KPI")


def compute_multi_kpi_metrics(
    long_real: pd.DataFrame,
    long_syn: pd.DataFrame,
    kpi_list: list[str],
    n_projections: int = 200,
) -> dict:
    """Cross-KPI / joint-distribution similarity metrics, real vs synthetic."""
    real_wide = long_to_wide(long_real[long_real["kpi_id"].isin(kpi_list)])
    syn_wide = long_to_wide(long_syn[long_syn["kpi_id"].isin(kpi_list)])

    # A KPI absent from one side's windows (e.g. not generated/observed at all
    # for this config) has no pivoted column there — restrict to the shared set.
    shared_kpis = [k for k in kpi_list if k in real_wide.columns and k in syn_wide.columns]

    real_complete = real_wide[shared_kpis].dropna().to_numpy(dtype=float)
    syn_complete = syn_wide[shared_kpis].dropna().to_numpy(dtype=float)

    corr_r = pairwise_corr_matrix_from_pdf(real_wide, shared_kpis)
    corr_s = pairwise_corr_matrix_from_pdf(syn_wide, shared_kpis)

    p_r = partial_correlations(real_complete)
    p_s = partial_correlations(syn_complete)

    return {
        "n_kpis": len(shared_kpis),
        "n_kpis_dropped": len(kpi_list) - len(shared_kpis),
        "real_complete_rows": real_complete.shape[0],
        "syn_complete_rows": syn_complete.shape[0],
        "sliced_wasserstein": metric_sliced_wasserstein(real_complete, syn_complete, n_projections=n_projections),
        "mmd_multivariate": metric_mmd_multivariate(real_complete, syn_complete),
        "pairwise_corr_dist": corr_matrix_distance(corr_r, corr_s),
        "partial_corr_dist": partial_corr_distance(p_r, p_s),
    }


## Run metrics per config

Generates the same real/synthetic windows as `run_visual_checks.ipynb` (mapped back
to physical KPI units), then computes the metric tables instead of plotting them.

Comparable across KPIs (normalized/bounded, so you can scan the column and compare magnitudes between different KPIs):

- jensen_shannon — bounded in [0,1]. 0 = identical marginal densities, 1 = no overlap at all. This is your best "scan first" column. 
- mmd — kernel distance with an adaptive (median-heuristic) bandwidth, so it's roughly scale-invariant too, just not hard-bounded to [0,1]. E.g. Values under ~0.05 are "close," above ~0.3 are clearly different distributions.
- acf_dist — L2 distance between two ACF curves (each value in [-1,1]), normalized by lag count. Small values mean the temporal correlation structure (e.g. daily rhythm) is preserved.
- ls_spectrum_dist — L2 distance between two normalized power spectra (sum to 1), max possible is √2. Small = same periodic structure (daily/weekly cycles line up).


Not comparable across KPIs (raw KPI units, so a "697" isn't worse than a "0.04" — it just means that KPI's physical scale is bigger):

- wasserstein — cost to morph one value distribution into the other, in the KPI's native units.
- hourly_rmse — RMSE between the two average-by-hour-of-day profiles, same native units. Useful to compare within a KPI against its own wasserstein: if - - hourly_rmse is much smaller than wasserstein, the diurnal shape is fine and the gap is elsewhere (spread/tails); if they're close, the mismatch is mostly a time-of-day problem.


KPIs with NaNs should not be analzyed

In [ ]:
N_WINDOWS = 8       # real windows / synthetic weeks sampled per config
ANCHOR_DATE = "2024-01-15"
MAX_LAG = 48         # ACF lag (hours) for the single-KPI table

single_kpi_tables = {}
multi_kpi_summaries = {}

for label, cid in CONFIGS.items():
    print(f"\n========== {label}  (cell_id={cid}) ==========")

    X_real, _, anchors_real = select_real_windows(cid, max_windows=N_WINDOWS)
    long_real_scaled = wide_timespan_to_long(wide_from_array(X_real, anchors_real, cid))

    syn_wide = generate_windows(
        model=model,
        config_encoder=config_encoder,
        cell_config_map=cell_config_map,
        cell_id=cid,
        anchor_date=ANCHOR_DATE,
        n_weeks=N_WINDOWS,
        holiday=0,
        batch_size=64,
        seed=0,
        kpi_list=kpi_columns,
    )
    long_syn_scaled = wide_timespan_to_long(syn_wide)

    # Map both real and synthetic back to original, physical KPI units before comparing.
    long_real, long_syn = inverse_to_physical(long_real_scaled, long_syn_scaled, cid)

    single_kpi_tables[label] = compute_single_kpi_metrics(long_real, long_syn, EXAMPLE_KPIS, max_lag=MAX_LAG)
    multi_kpi_summaries[label] = compute_multi_kpi_metrics(long_real, long_syn, EXAMPLE_KPIS)

    display(single_kpi_tables[label])
    display(pd.Series(multi_kpi_summaries[label], name=label))


## Combined summary across configs

In [ ]:
combined_single = pd.concat(single_kpi_tables, names=["config", "KPI"])
display(combined_single)

multi_summary_df = pd.DataFrame(multi_kpi_summaries).T
display(multi_summary_df)
